In [ ]:
from pathlib import Path
import re
import numpy as np
import h5py
import matplotlib.pyplot as plt


# ----------------------------
# 1) Find the most recent ARTIQ result file
# ----------------------------
def find_latest_artiq_h5(results_root=None, name_regex=None):
    """
    results_root: Path to ARTIQ results (defaults to ~/.local/share/artiq/results)
    name_regex: optional regex to filter files (e.g., r"EntanglerDatasetTest")
    """

    results_root = Path("/home/jrydberg/Documents/Projects/Artiq_envs/madmax-artiq-env/entagnler/results")

    if not results_root.exists():
        raise FileNotFoundError(f"Results folder not found: {results_root}")

    h5_files = list(results_root.rglob("*.h5"))
    if name_regex:
        rx = re.compile(name_regex)
        h5_files = [p for p in h5_files if rx.search(p.name)]

    if not h5_files:
        raise FileNotFoundError(f"No .h5 files found under {results_root}")

    latest = max(h5_files, key=lambda p: p.stat().st_mtime)
    return latest


latest_h5 = find_latest_artiq_h5(name_regex=None)  # e.g. name_regex=r"Entangler"
print("Latest H5:", latest_h5)


# ----------------------------
# 2) Read datasets from HDF5
# ----------------------------
def read_dataset(h5f, key):
    """
    Reads /datasets/<key> from ARTIQ HDF5 result file.
    Returns Python-native objects (lists/ints/floats) where possible.
    """
    ds_path = f"datasets/{key}"
    if ds_path not in h5f:
        return None
    obj = h5f[ds_path][()]

    # Convert numpy scalars to python
    if isinstance(obj, (np.generic,)):
        return obj.item()

    # Convert numpy arrays to lists
    if isinstance(obj, np.ndarray):
        return obj.tolist()

    return obj


with h5py.File(latest_h5, "r") as f:
    print("Available datasets in HDF5:", list(f["datasets"].keys()))
    success_rate = read_dataset(f, "entangler.success_rate")
    status_hist  = read_dataset(f, "entangler.status_hist")
    reason_hist  = read_dataset(f, "entangler.reason_hist")
    end_ts_hist  = read_dataset(f, "entangler.end_ts_hist")
    ts_hist      = read_dataset(f, "entangler.ts_hist")

print("Loaded datasets:",
      {k: (None if v is None else f"type={type(v).__name__}, len={len(v) if hasattr(v,'__len__') else 'scalar'}")
       for k, v in [
           ("success_rate", success_rate),
           ("status_hist", status_hist),
           ("reason_hist", reason_hist),
           ("end_ts_hist", end_ts_hist),
           ("ts_hist", ts_hist),
       ]})


# ----------------------------
# 3) Sanity + normalize to numpy
# ----------------------------
def as_np_int(x, name):
    if x is None:
        raise KeyError(f"Missing dataset: entangler.{name}")
    return np.array(x, dtype=np.int64)

status = as_np_int(status_hist, "status_hist")
reason = as_np_int(reason_hist, "reason_hist")
end_ts = as_np_int(end_ts_hist, "end_ts_hist")
ts = np.array(ts_hist, dtype=np.int64)  # shape (N,4) expected

N = len(status)
assert reason.shape[0] == N and end_ts.shape[0] == N, "Length mismatch among histories"
assert ts.ndim == 2 and ts.shape[0] == N, "ts_hist should be shape (N,4-ish)"

print(f"\nN shots: {N}")
print("ts_hist shape:", ts.shape)


# ----------------------------
# 4) Compute useful derived metrics
# ----------------------------
# Your demo code treats status bit 0b010 as "succeeded"
success = (status & 0b010) != 0
success_rate_calc = success.mean()

# Which inputs fired, based on timestamps > 0 (may vary by firmware, but good heuristic)
fired = ts > 0
fired_mask = (fired[:, 0].astype(np.int64) << 0) | (fired[:, 1].astype(np.int64) << 1) | \
             (fired[:, 2].astype(np.int64) << 2) | (fired[:, 3].astype(np.int64) << 3)

print("\n=== Summary ===")
print("success_rate dataset:", success_rate)
print("success_rate (status&0b010):", float(success_rate_calc), f"({success.sum()}/{N})")

# histograms (counts)
def counts(arr):
    u, c = np.unique(arr, return_counts=True)
    return dict(zip(u.tolist(), c.tolist()))

print("\nReason counts:", counts(reason))
print("Status counts:", {f"{k} (0b{k:b})": v for k, v in counts(status).items()})

mask_counts = counts(fired_mask)
print("\nFired-mask counts (mask -> count):", {f"0b{k:04b}": v for k, v in mask_counts.items()})

# Per-input firing rate
for i in range(min(4, ts.shape[1])):
    print(f"input{i} fired fraction:", float(fired[:, i].mean()))


# ----------------------------
# 5) Quick plots
# ----------------------------
# (A) Success over shots
plt.figure()
plt.plot(success.astype(int))
plt.xlabel("shot index")
plt.ylabel("success (status&0b010)")
plt.title("Entangler success per shot")
plt.show()

# (B) Status histogram
plt.figure()
u, c = np.unique(status, return_counts=True)
plt.bar([str(int(x)) for x in u], c)
plt.xlabel("status value")
plt.ylabel("count")
plt.title("Status histogram")
plt.show()

# (C) Fired mask histogram
plt.figure()
u, c = np.unique(fired_mask, return_counts=True)
labels = [f"0b{int(x):04b}" for x in u]
plt.bar(labels, c)
plt.xlabel("fired mask (inputs 3..0)")
plt.ylabel("count")
plt.title("Input fired-mask histogram")
plt.show()

# (D) Timestamp scatter for each input (only where ts>0)
for i in range(min(4, ts.shape[1])):
    plt.figure()
    idx = np.where(ts[:, i] > 0)[0]
    plt.scatter(idx, ts[idx, i], s=10)
    plt.xlabel("shot index")
    plt.ylabel(f"timestamp_mu input{i}")
    plt.title(f"Timestamps for input{i} (ts>0)")
    plt.show()


Latest H5: /home/jrydberg/Documents/Projects/Artiq_envs/madmax-artiq-env/entagnler/results/2026-02-13/11/000000024-EntanglerDemo.h5
Loaded datasets: {'success_rate': None, 'status_hist': None, 'reason_hist': None, 'end_ts_hist': None, 'ts_hist': None}


KeyError: 'Missing dataset: entangler.status_hist'